<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/ndvi_lst_sharpening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade xee

In [2]:
!pip install -U geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 73.6 MB/s  0:00:00


In [3]:
import ee

In [6]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url='https://earthengine-highvolume.googleapis.com'
)

In [7]:
import geemap

In [8]:
map = geemap.Map(basemap="SATELLITE")
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [9]:
roi = map.draw_last_feature.geometry()
roi

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "Feature.geometry",
    "arguments": {
      "feature": {
        "functionInvocationValue": {
          "functionName": "Feature",
          "arguments": {
            "geometry": {
              "functionInvocationValue": {
                "functionName": "GeometryConstructors.Polygon",
                "arguments": {
                  "coordinates": {
                    "constantValue": [
                      [
                        [
                          30.124518,
                          5.768865
                        ],
                        [
                          30.124518,
                          7.656383
                        ],
                        [
                          32.530507,
                          7.656383
                        ],
                        [
                          32.530507,
                          5.768865
                        ],
                        [
                          30.124518,
                          5.768865
                        ]
                      ]
                    ]
                  },
                  "geodesic": {
                    "constantValue": false
                  }
                }
              }
            }
          }
        }
      }
    }
  }
})

In [21]:
landsat = (ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
          .filterDate("2025", "2026")
          .filterBounds(roi)
)
landsat

In [22]:
def ndvi_lst(img):
  qa_band = img.select("QA_PIXEL")
  bit1 = qa_band.bitwiseAnd(1 << 1).neq(0)
  bit2 = qa_band.bitwiseAnd(1 << 2).neq(0)
  bit3 = qa_band.bitwiseAnd(1 << 3).neq(0)
  bit4 = qa_band.bitwiseAnd(1 << 4).neq(0)
  mask = bit1.Or(bit2).Or(bit3).Or(bit4)
  sr_bands = img.select("SR_B.*").multiply(2.75e-05).add(-0.2)
  ndvi = sr_bands.normalizedDifference(["SR_B5", "SR_B4"]).rename("ndvi")
  lst = img.select("ST_B10").multiply(0.00341802).add(149).rename("lst")
  stack = ee.Image.cat([ndvi, lst])
  return stack.updateMask(mask.Not()).copyProperties(img, ["system:time_start"])


In [23]:
landsat_ndvi_lst = (landsat.map(ndvi_lst)
)

landsat_ndvi_lst

## **Convert the image to Xarray**

In [24]:
from xee import helpers
from shapely.geometry import shape
import xarray as xr

In [25]:
landsat.first().select("SR_B1").projection()

In [26]:
grid_30m = helpers.fit_geometry(
    geometry = shape(roi.getInfo()),
    grid_crs = "EPSG:32636",
    grid_scale = (30, -30)
)

In [28]:
ds = xr.open_dataset(
    landsat_ndvi_lst,
    engine = "ee",
    **grid_30m,
    chunks = "auto"
)
ds

<xarray.Dataset> Size: 80GB
Dimensions:  (time: 160, y: 6991, x: 8891)
Coordinates:
  * time     (time) datetime64[ns] 1kB 2025-01-04T08:05:36.041000 ... 2025-12...
  * y        (y) float64 56kB 8.474e+05 8.473e+05 ... 6.377e+05 6.377e+05
  * x        (x) float64 71kB 1.815e+05 1.815e+05 ... 4.482e+05 4.482e+05
Data variables:
    ndvi     (time, y, x) float32 40GB dask.array<chunksize=(96, 512, 512), meta=np.ndarray>
    lst      (time, y, x) float32 40GB dask.array<chunksize=(96, 512, 512), meta=np.ndarray>

In [ ]:
from dask.diagnostics import ProgressBar
with ProgressBar():
  ds = ds.compute()

[                                        ] | 0% Completed | 2.86 s

[                                        ] | 0% Completed | 6.62 s

[                                        ] | 0% Completed | 9.96 s

[                                        ] | 0% Completed | 17.95 s

[                                        ] | 0% Completed | 18.15 s

[                                        ] | 1% Completed | 21.07 s

... (61 outputs left out)

[######                                  ] | 17% Completed | 302.14 s

[######                                  ] | 17% Completed | 302.27 s

[#######                                 ] | 17% Completed | 305.35 s

[#######                                 ] | 17% Completed | 309.28 s

In [2]:
ds = ds.sortby("time")

NameError: name 'ds' is not defined